## Port AbacusSummit `base` halos → ltu-cmass format (with NFW-fit concentration)

Loads the **cleaned CompaSO** halo catalogs at z=0.5 out of the 15 TB HTAR archive
`z0.500_base.tar`, grabs pos / vel / mass / **concentration**, applies the CHARM mass cut,
and writes `halos.h5` in the same layout as `port_quijote.ipynb`.

All the logic lives in **`port_abacus_lib.py`** (imported below); this notebook is the
interactive / single-sim front-end. The full grid is run as a **SLURM array** — see the last cell.

### Concentration: NFW profile fit → c200c
We fit an NFW profile `M(<r) = A·μ(r/rs)`, `μ(x)=ln(1+x)−x/(1+x)`, in log-space to the
enclosed-mass radii `r10…r98_L2com` (about the L2 center), then solve the overdensity
condition for R200c → **c = R200c / rs**. This matches Quijote's Rockstar `R200c/Rs` definition.

> We do **not** use the simpler `2.1626·SO_radius/rvcirc_max`: it mixes the whole-L1 SO radius
> with the L2 core and produces a spurious *multimodal* concentration distribution driven by
> substructure. The NFW fit is unimodal and agrees with Quijote at comparable cosmology.
> See `/home/x-mho1/git/ltu-gobig-notes/experiments/2026-08-07_abacus_nfw_concentration_port/`
> for the full writeup (the exploratory decomposition/benchmark scripts used to diagnose this
> were scratch work and are no longer in this repo).

### How to run
1. `conda activate /anvil/scratch/x-mho1/envs/abacusport` (kernel **abacusport**).
2. Run top to bottom for a single test sim. `get_index()` loads a cached tar index
   (`z0.500_base.index.pkl`, built once ~3 min).
3. **For the full grid, do NOT run in-notebook** — the login node kills processes at a ~10-min
   CPU cap. Use `sbatch run_array.sh` (compute nodes). See the final cell.

### Data notes
- 114 `base` sims (`c001_ph000`…`c181_ph000`); LHID→SimName via `abacus_custom_table.csv`.
- HTAR `.idx` byte offsets are *logical* (not seekable) → we use Python `tarfile` random access.
- Per sim ≈ 90 GB of `halo_info` pulled from the tar; slabs are read **one at a time**
  (bounded memory) then deleted.

In [ ]:
# All porting logic lives in port_abacus_lib.py (shared with port_abacus_one.py / the SLURM array)
import h5py, numpy as np
from os.path import join
from port_abacus_lib import (
    get_index, process_lhid, load_and_fit, extract_sim, fit_c200c,
    SIMNAME, OUT_TMPL, TAR, INDEX_PKL, MMIN, A, Z,
)
print(f'{len(SIMNAME)} sims in table (LHID {min(SIMNAME)}..{max(SIMNAME)}); '
      f'e.g. 6 -> {SIMNAME[6]}, 118 -> {SIMNAME[118]}')
print(f'output template : {OUT_TMPL}')
print(f'mass cut        : {MMIN:.0e} Msun/h   z={Z}  a={A:.6f}')

In [ ]:
# Random-access index of the tar. Loads the cached pickle if present, else scans once (~3 min)
# and pickles it so the SLURM array tasks don't each rescan 15 TB.
tf_dict, index = get_index()
print(f'{len(index)} halo_info + cleaned_halo_info members indexed  (cache: {INDEX_PKL})')

In [ ]:
# --- single-sim test (interactive). For the full grid use the SLURM array (last cell). ---
# NOTE: extract + slab-by-slab read + NFW fit is ~20-25 min and ~90 GB temp; on the login node
# this WILL be killed at the ~10-min CPU cap. Only run here on a compute node / interactive alloc.
from time import time
lhid = 6
t0 = time()
print(process_lhid(lhid, index, tf_dict, overwrite=True, verbose=True), f'({time()-t0:.0f}s)')

In [ ]:
# --- inspect a ported catalog ---
lhid = 6
with h5py.File(join(OUT_TMPL.format(lhid=lhid), 'halos.h5'), 'r') as f:
    g = f[f'{A:.6f}']
    pos, vel = g['pos'][:], g['vel'][:]
    mass, conc = g['mass'][:], g['concentration'][:]
print(f'lhid {lhid} ({SIMNAME[lhid]}): {len(mass)} halos')
print(f'  pos  [Mpc/h]  min {pos.min(0)}  max {pos.max(0)}')
print(f'  vel  [km/s]   |max| {np.abs(vel).max():.0f}')
print(f'  logM [Msun/h] {mass.min():.2f} .. {np.median(mass):.2f} .. {mass.max():.2f}')
print(f'  c200c         {conc.min():.2f} .. {np.median(conc):.2f} .. {conc.max():.2f}'
      f'   (finite={np.isfinite(conc).mean()*100:.1f}%)')

## Full grid → SLURM array (production)

The login node kills processes at a ~10-min CPU cap, and the full grid is ~114 sims × ~20 min,
so run it on compute nodes via the array driver:

- **`port_abacus_one.py <lhid>`** — ports one sim (loads the cached tar index, extracts its slabs,
  slab-by-slab cleaned read + NFW `c200c` fit, writes `halos.h5` + `config.yaml`, deletes the temp).
  Resumable.
- **`run_array.sh`** — the SLURM array wrapper (`phy240043`, `shared`, one sim per task).

```bash
# 1. build the tar index once (~3 min, login node OK) — array tasks then load the pickle
python -c "from port_abacus_lib import get_index; get_index()"

# 2. validate one task end-to-end on a compute node
sbatch --array=6 run_array.sh

# 3. launch the whole grid (8 concurrent tasks; ~90 GB scratch temp each)
sbatch run_array.sh                 # --array=0-118%8 is set in the script
```

Concurrency is capped at `%8` because each task holds ~90 GB of extracted slabs (deleted on
completion). Tasks skip any lhid whose `halos.h5` already exists (`--overwrite` to force);
lhids absent from either source tar are skipped automatically.

**Output:** `abacus/nbodyall/L2000-N256/{lhid}/halos.h5`, group `0.666667`, datasets
`pos` (cMpc/h), `vel` (km/s), `mass` (log10 Msun/h), `concentration` (NFW `c200c`), plus a
provenance `config.yaml` (meta+nbody sections only — this port doesn't run bias/HOD/survey).

`abacus/nbodyconc` is a second view of the same data: symlinks using the **same lhid** as
`abacus/custom` (both derive from `abacus_custom_table.csv`), so it's a drop-in companion to
`custom` that adds the NFW-fit concentration field, not a different indexing scheme. See
`mirror_custom_ordering.py`.